# Inspect spectral image metadata in CZI and OME-TIFF

Run in Python 3.12 with `lumic[examples]==0.1.0rc2` installed. The first run
downloads the high-SNR COS-7 CZI (about 145 MB) from
[BioImage Archive S-BIAD4448](https://www.ebi.ac.uk/biostudies/bioimages/studies/S-BIAD4448).
Later runs reuse the cached CZI.

We inspect the CZI header first, then load one plane to create an OME-TIFF
with Lumic. We inspect the exported header without reading its pixels.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

from lumic.data import download_cos7_data
from lumic.io import load_image, open_image

In [ ]:
czi_path, = download_cos7_data(snr="high", progress=True)

## 1. Inspect the CZI header

`open_image()` and `summary()` report the available layout, detector channel
count, and calibration before loading any pixels.

| Metadata | How to read it |
| --- | --- |
| `axes`, `shape` | Array dimensions and their sizes in the same order. `C` is detector channel, `Y/X` are spatial axes, `Z` is depth, and `T` is time. |
| `native_axes`, `dimension_ranges` | The reader's native axis order and its index bounds, written as start-inclusive, end-exclusive pairs. Native axes may include dimensions omitted from Lumic's array. |
| `pixel_sizes_um` | Physical pixel spacing in micrometres for each calibrated spatial axis. |
| `wavelengths_nm` | A wavelength coordinate for each detector channel, when available. |
| `detection_bands_nm` | A lower and upper wavelength bound for each detector channel, when available. |

Each detector channel measures light in a wavelength range and can contain
signal from several fluorophores.

In [ ]:
with open_image(czi_path) as czi_reader:
    print(czi_reader.summary())
    print("Native axes:", czi_reader.native_axes)
    print("Dimension ranges:", czi_reader.dimension_ranges)
    print("First five wavelength coordinates (nm):", czi_reader.wavelengths_nm[:5])
    print("First five detector bands (nm):")
    print(czi_reader.detection_bands_nm[:5])

## 2. Export one plane and inspect the OME-TIFF header

Creating the OME-TIFF requires reading the selected CZI plane. The image's
`provenance` records how Lumic obtained it, including the source format,
native axes, plane selection when applicable, and spatial origin.

The export stores pixel calibration in standard OME fields and preserves
the detector wavelengths and bands in a Lumic OME annotation. The temporary
file is removed when the cell finishes. The header fields used below remain
available on the closed reader.

In [ ]:
image = load_image(czi_path, t=0, z=0)
print("Source provenance:", image.provenance)

with TemporaryDirectory() as output_dir:
    ome_path = Path(output_dir) / "cos7-high-snr.ome.tiff"
    image.save_ome_tiff(ome_path)

    with open_image(ome_path) as ome_reader:
        print(ome_reader.summary())
        print("Image kind:", ome_reader.kind)
        print("Wavelength source:", ome_reader.wavelength_source)

## 3. Check the metadata after export

Compare the OME-TIFF header to the selected CZI plane. This checks the
array layout, physical spacing, and per-channel wavelength metadata.

In [ ]:
assert ome_reader.axes == image.axes
assert ome_reader.shape == image.data.shape
assert ome_reader.kind == image.kind

assert ome_reader.pixel_sizes_um.keys() == image.pixel_sizes_um.keys()
for axis, spacing_um in image.pixel_sizes_um.items():
    np.testing.assert_allclose(ome_reader.pixel_sizes_um[axis], spacing_um)

np.testing.assert_array_equal(ome_reader.wavelengths_nm, image.wavelengths_nm)
np.testing.assert_array_equal(ome_reader.detection_bands_nm, image.detection_bands_nm)
print("The export preserved layout, image kind, calibration, and detector coordinates.")

## How the metadata can vary across files

- For CZI files with ordered, non-overlapping windows of positive width,
  Lumic uses the window midpoints as wavelength coordinates. Overlapping
  windows can have detection bounds with `wavelengths_nm=None`.
- For other OME-TIFF exports, Lumic can read wavelength coordinates from
  complete emission-wavelength fields or numeric channel labels such as
  `415 nm`. Channel labels may be rounded; missing detector bounds remain
  `None`. Check `wavelength_source` to see where the coordinates came from.
- A Lumic export preserves the metadata described in the
  [I/O guide](../../lumic/IO_README.md). Future work will extend CZI-to-OME-TIFF
  export to preserve more acquisition settings, such as detector gain and
  scan averaging, where those settings are available. Keep the original CZI
  for the full instrument settings and vendor metadata.

See [minimal_load.ipynb](minimal_load.ipynb) to reload the exported pixels
and compare them with the source.